# PromptForge Phase 3 — Combined Pipeline + Evaluation + Space

Wires Phase 1 (Quality) + Phase 2 (Optimizer) end-to-end.

```text
Prompt
  → Quality Scorer
  → Optimizer
  → Re-score optimized prompt
  → Before/After comparison + eval report
  → Gradio / HF Space
```

**Runtime → GPU** recommended (both models in memory).

This notebook assumes you already have:
- `/content/promptforge-quality-model` (Phase 1)
- `/content/promptforge-optimizer-model` (Phase 2)

Or set the paths below to your Drive / Hub repos.


In [ ]:
!pip install -q \
    transformers \
    datasets \
    accelerate \
    peft \
    scikit-learn \
    pandas \
    numpy \
    scipy \
    huggingface_hub \
    gradio \
    pyyaml


In [ ]:
import os
import json
from pathlib import Path

import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

QUALITY_MODEL = "/content/promptforge-quality-model"
OPTIMIZER_MODEL = "/content/promptforge-optimizer-model"

# Optional: Hub ids after you publish
# QUALITY_MODEL = "YOUR_USER/PromptForge-Quality"
# OPTIMIZER_MODEL = "YOUR_USER/PromptForge-Optimizer"

print("Quality:", QUALITY_MODEL)
print("Optimizer:", OPTIMIZER_MODEL)


## Install / import PromptForge package


In [ ]:
import os
from pathlib import Path

REPO_DIR = Path("/content/promptModel")
if (Path.cwd() / "src" / "promptforge").exists():
    REPO_DIR = Path.cwd()

os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)

!pip install -q -e ".[demo]"

import promptforge
print("promptforge", promptforge.__version__)


## Combined pipeline


In [ ]:
from promptforge import PromptForge

pf = PromptForge(
    quality_model_path=QUALITY_MODEL,
    optimizer_model_path=OPTIMIZER_MODEL,
    prefer_gpu=True,
)

result = pf.run(
    "Build me a website for a startup.",
    task_type="coding",
    rescore_optimized=True,
)

print(json.dumps(result, indent=2))


In [ ]:
test_prompts = [
    "Make an app.",
    "Build me a website.",
    "Write something about AI.",
    "Make a Python API for beginners.",
    "Create a workout plan.",
]

for prompt in test_prompts:
    out = pf.run(prompt, task_type="general", rescore_optimized=True)
    print("=" * 80)
    print("ORIGINAL:", prompt)
    print(
        f"SCORE: {out['before']['quality_score']} → {out['after']['quality_score']} "
        f"(Δ {out['delta']['quality_score']})"
    )
    print("CHANGES:", ", ".join(out["changes"]))
    print("OPTIMIZED:\n", out["optimized_prompt"][:500], "...")


## Evaluation (score lift + preservation + downstream proxy)


In [ ]:
from promptforge.evaluation import (
    run_pipeline_evaluation,
    run_downstream_proxy_eval,
    save_eval_report,
)

pipeline_report = run_pipeline_evaluation(pf, task_type="general")
downstream_report = run_downstream_proxy_eval(pf, task_type="general")

payload = {
    "pipeline": pipeline_report,
    "downstream_proxy": downstream_report,
}

out_path = "/content/phase3_pipeline_report.json"
save_eval_report(payload, out_path)

print("Pipeline summary:")
print(json.dumps(pipeline_report["summary"], indent=2))
print("\nDownstream proxy summary:")
print(json.dumps(downstream_report["summary"], indent=2))
print("\nSaved:", out_path)


## Before / After table (HF card style)


In [ ]:
import pandas as pd

rows = []
for row in pipeline_report["rows"]:
    rows.append({
        "prompt": row["prompt"][:60],
        "before": row["before_score"],
        "after": row["after_score"],
        "delta": row["score_delta"],
        "improved": row["improved"],
    })

display(pd.DataFrame(rows))

summary = pipeline_report["summary"]
lift = downstream_report["summary"]["relative_lift_pct"]
print("PromptForge")
print("-" * 30)
print(f"Quality         {summary['mean_before_score']:.1f} -> {summary['mean_after_score']:.1f}")
print(f"Improved        {summary['pct_improved']:.1f}% of prompts")
print(f"Downstream lift {lift:.1f}%")


## Launch Gradio demo (HF Space style)


In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR / "demo"))

from app import build_demo

demo = build_demo(
    quality_model_path=QUALITY_MODEL,
    optimizer_model_path=OPTIMIZER_MODEL,
    prefer_gpu=True,
)

# In Colab, share=True gives a public link
demo.launch(share=True)


## Export / publish Space

```bash
python scripts/export_space.py --out outputs/hf_space
```

Then create a Hugging Face Space (Gradio), upload that folder, and set:
- `PROMPTFORGE_QUALITY_MODEL`
- `PROMPTFORGE_OPTIMIZER_MODEL`
